<center><h1>Sentiment classification: TF-IDF n-grams vs. spaCy embeddings</h1></center>

How can we compare a **count-based n-gram representation** (TF-IDF with unigrams+bigrams) with a
**pretrained word-embedding representation** (spaCy) for the same task? In this notebook we build a
sentiment classifier for app reviews using both representations, feed them to the **same neural network
classifier** (built with TensorFlow/Keras), and compare the results.

In [11]:
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 64.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [12]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import spacy
import pandas as pd
import numpy as np
import re
import tensorflow as tf

nlp = spacy.load('en_core_web_md')
tf.random.set_seed(42)

In [2]:
from google.colab import files
uploaded = files.upload()

dataset = pd.read_csv('all_data.csv')
dataset = dataset.dropna(subset=['review', 'sentiment']).reset_index(drop=True)
dataset.head()

Saving all_data.csv to all_data.csv


,review,sentiment
0,Aditya Ingole Deaf,2
1,I love the app.! There is no issue but if u co...,1
2,"So hard to use. The web app failed, and the mo...",0
3,I hate that the app makes a sound every time s...,1
4,Useless at BSE star MF meet.voice too mych slo...,0


In [14]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

dataset['clean_text'] = dataset['review'].map(preprocess_text)
dataset = dataset[dataset['clean_text'].str.len() > 0].reset_index(drop=True)
dataset.shape

(39804, 3)

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    dataset['clean_text'], dataset['sentiment'], test_size=0.2, random_state=42, stratify=dataset['sentiment']
)
len(X_train), len(X_test)

(31843, 7961)

## Representation 1: Bag of N-Grams with TF-IDF

Instead of only unigrams, we include bigrams (`ngram_range=(1, 2)`) so the vectorizer also captures short
phrases like "not good" or "keeps crashing", which a single-word bag of words would miss.

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_ngrams = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
X_train_ngrams = tfidf_ngrams.fit_transform(X_train).toarray().astype('float32')
X_test_ngrams = tfidf_ngrams.transform(X_test).toarray().astype('float32')
X_train_ngrams.shape

(31843, 5000)

## Representation 2: spaCy word embeddings

Same idea we used in the previous notebook: average the pretrained word vectors of every token in the sentence.

In [7]:
def get_longest_text(texts):
    return max(len(t.split()) for t in texts)

longest_input = get_longest_text(dataset['clean_text'])
print(f"Longest review is {longest_input} tokens")

Longest review is 468 tokens


In [17]:
nlp = spacy.load("en_core_web_md")

def get_avg_vectors(texts, nlp_model):
    return np.array(
        [doc.vector for doc in nlp_model.pipe(texts, batch_size=256)],
        dtype="float32"
    )

X_train_emb = get_avg_vectors(X_train, nlp)
X_test_emb = get_avg_vectors(X_test, nlp)
print(X_train_emb.shape)
print(X_test_emb.shape)


(31843, 300)
(7961, 300)


## Encoding the labels

`sentiment` is already an integer (0 = negative, 1 = neutral, 2 = positive), so we keep it as it is and just
remember the mapping for later. We still fit a `LabelEncoder` so we can use `encoder.classes_` when reporting
predictions, the same way the original n-grams notebook used its word encoder.

In [18]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
encoder.fit([0, 1, 2])
y_train_arr = np.array(y_train)
y_test_arr = np.array(y_test)
encoder.classes_

array([0, 1, 2])

## Build the model

We keep the model architecture identical for both representations: a small feed-forward network built with
`tf.keras`, so the comparison is only about the *input representation*, not the model.

In [19]:
def build_model(input_dim, num_classes=3):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [20]:
model_ngrams = build_model(input_dim=X_train_ngrams.shape[1])
history_ngrams = model_ngrams.fit(X_train_ngrams, y_train_arr, epochs=8, batch_size=128,
                                   validation_split=0.1, verbose=0)
print("Final training accuracy:", history_ngrams.history['accuracy'][-1])

Final training accuracy: 0.8847442269325256


In [21]:
model_emb = build_model(input_dim=X_train_emb.shape[1])
history_emb = model_emb.fit(X_train_emb, y_train_arr, epochs=8, batch_size=128,
                             validation_split=0.1, verbose=0)
print("Final training accuracy:", history_emb.history['accuracy'][-1])

Final training accuracy: 0.5702072978019714


## Evaluate the model

In [22]:
from sklearn.metrics import f1_score, classification_report

test_loss_ngrams, test_acc_ngrams = model_ngrams.evaluate(X_test_ngrams, y_test_arr, verbose=0)
pred_ngrams = np.argmax(model_ngrams.predict(X_test_ngrams, verbose=0), axis=1)
f1_ngrams = f1_score(y_test_arr, pred_ngrams, average='macro')
print(f"TF-IDF (n-grams) -> accuracy: {test_acc_ngrams:.4f}, macro F1: {f1_ngrams:.4f}")
print(classification_report(y_test_arr, pred_ngrams))

TF-IDF (n-grams) -> accuracy: 0.6891, macro F1: 0.6756
              precision    recall  f1-score   support

           0       0.75      0.70      0.72      2657
           1       0.62      0.51      0.56      2302
           2       0.69      0.82      0.75      3002

    accuracy                           0.69      7961
   macro avg       0.68      0.68      0.68      7961
weighted avg       0.69      0.69      0.68      7961



In [23]:
test_loss_emb, test_acc_emb = model_emb.evaluate(X_test_emb, y_test_arr, verbose=0)
pred_emb = np.argmax(model_emb.predict(X_test_emb, verbose=0), axis=1)
f1_emb = f1_score(y_test_arr, pred_emb, average='macro')
print(f"spaCy embeddings -> accuracy: {test_acc_emb:.4f}, macro F1: {f1_emb:.4f}")
print(classification_report(y_test_arr, pred_emb))

spaCy embeddings -> accuracy: 0.5644, macro F1: 0.5316
              precision    recall  f1-score   support

           0       0.56      0.66      0.60      2657
           1       0.45      0.26      0.33      2302
           2       0.62      0.71      0.66      3002

    accuracy                           0.56      7961
   macro avg       0.54      0.54      0.53      7961
weighted avg       0.55      0.56      0.55      7961



Let's make a prediction function

In [24]:
encoder.classes_[2]

np.int64(2)

In [25]:
text = "the video quality is really bad and it keeps freezing"

clean = preprocess_text(text)
vec = tfidf_ngrams.transform([clean]).toarray().astype('float32')
proba_ngrams = model_ngrams.predict(vec, verbose=0)
for label, prob in zip(encoder.classes_, proba_ngrams[0]):
    print(label, ':', round(float(prob), 3))

0 : 0.842
1 : 0.002
2 : 0.156


In [26]:
def predict_ngrams(text):
    clean = preprocess_text(text)
    vec = tfidf_ngrams.transform([clean]).toarray().astype('float32')
    pred = np.argmax(model_ngrams.predict(vec, verbose=0), axis=1)[0]
    return {0: 'negative', 1: 'neutral', 2: 'positive'}[pred]

def predict_embeddings(text):
    clean = preprocess_text(text)
    vec = get_avg_vectors([clean], nlp)
    pred = np.argmax(model_emb.predict(vec, verbose=0), axis=1)[0]
    return {0: 'negative', 1: 'neutral', 2: 'positive'}[pred]

predict_ngrams("love this app so much, works perfectly")

'positive'

In [27]:
predict_embeddings("love this app so much, works perfectly")

'positive'

## Comparison summary

Both models share the exact same architecture (`Dense(128) -> Dropout -> Dense(64) -> Dense(3, softmax)`)
and the exact same preprocessing; only the input representation changes between the two runs. Adding bigrams
to TF-IDF gave it a bit more context than plain unigrams, and — as we saw in the previous notebook — the
sparse, vocabulary-specific TF-IDF representation was a better fit for this noisy short-text dataset than
the general-purpose pretrained embeddings, even with a neural network on top instead of a linear classifier.